### Definitions

In [114]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

PATH_EXAMPLE_CV = "./data/cv_example.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_MODEL_VECTOR_DIMENSIONS = 3072
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 3

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

In [115]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Return cosine similarities between a single query vector and a 3D matrix
    of shape (num_indices, size, embedding_dim).
    Zero-padded rows remain zero in the output.
    """
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def find_skill_compliance(
        skill_embedding: np.ndarray, 
        vector_matrix: np.ndarray, 
        threshold: float, 
        weight_matrix: np.ndarray = None, 
        weight: float = 0):
    
    cosine_similarities = cosine_similarities_matrix(skill_embedding, vector_matrix)
    binary_mask = (cosine_similarities > threshold).astype(np.int8)
    weights = []
    
    if weight != 0 and weight_matrix is not None:
        minimum_weights_matrix = np.where(binary_mask == 1, weight_matrix, 0)    
        binary_mask = ((weight >= minimum_weights_matrix) & (binary_mask == 1)).astype(np.int8)

    mappings = list(zip(*np.where(binary_mask != 0)))  
    match_count = len(mappings)
    weights = [weight_matrix[row, col] for row, col in mappings] if weight_matrix is not None else [0] * match_count
    return mappings, weights, match_count

def find_job_matches(mappings: list, skills_df: pd.DataFrame):
    positions_index = set([int(pair[0]) for pair in mappings])
    index_to_job_id = skills_df.drop_duplicates("index").set_index("index")["job_id"]
    return [str(index_to_job_id[i]) for i in positions_index]

### Get Candidate Info

In [116]:
RESUME_ID = "b6e8165a-b1af-4117-8a29-4a3a5fc95f32"
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

### Get Matching Market Info

In [117]:
matching_jobs_df = filter_job_postings(candidate_industries)
matching_jobs_ids = matching_jobs_df["id"].to_list()

market_hard_skills_df = get_position_skills(matching_jobs_ids, HARD_SKILLS_TABLE).sort_values("job_id")
market_soft_skills_df = get_position_skills(matching_jobs_ids, SOFT_SKILLS_TABLE).sort_values("job_id")

market_hard_skills_df["index"] = market_hard_skills_df.groupby("job_id").ngroup()
market_soft_skills_df["index"] = market_soft_skills_df.groupby("job_id").ngroup()

### Skills Matrixes

In [118]:
### STRING, VECTOR and WEIGHT SKills Matrixes
string_hard_skills_df = market_hard_skills_df[["index", "skill_description"]]
string_soft_skills_df = market_soft_skills_df[["index", "skill_description"]]
vector_hard_skills_df = market_hard_skills_df[["index", "embedding"]]
vector_soft_skills_df = market_soft_skills_df[["index", "embedding"]]
weight_hard_skills_df = market_hard_skills_df[["index", "weight"]]
weight_soft_skills_df = market_soft_skills_df[["index", "weight"]]

hard_skills_size = string_hard_skills_df.groupby('index').size().max()
soft_skills_size = string_soft_skills_df.groupby('index').size().max()

string_hard_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in string_hard_skills_df.groupby('index')
])
string_soft_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, soft_skills_size - len(group)), constant_values=0)
    for _, group in string_soft_skills_df.groupby('index')
])

vector_hard_skills_matrix = np.array([
    np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (hard_skills_size - len(group)))
    for _, group in vector_hard_skills_df.groupby('index')
], dtype=np.float32)
vector_soft_skills_matrix = np.array([
    np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (soft_skills_size - len(group)))
    for _, group in vector_soft_skills_df.groupby('index')
], dtype=np.float32)

weight_hard_skills_matrix = np.array([
    np.pad(group['weight'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in weight_hard_skills_df.groupby('index')
], dtype=np.float32)
weight_soft_skills_matrix = np.array([
    np.pad(group['weight'].values, (0, soft_skills_size - len(group)), constant_values=0)
    for _, group in weight_soft_skills_df.groupby('index')
], dtype=np.float32)

### Find market best matches

For each soft skill

In [119]:
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.69
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_soft_skills_df.shape[0] - 1

for i in range(0, skills_count):
    weight = candidate_soft_skills_df.iloc[(i, SOFT_SKILLS_WEIGHT_COLUMN_INDEX)] 
    skill_embedding = candidate_soft_skills_df.iloc[(i, SOFT_SKILLS_STRING_COLUMN_INDEX) ] 

    minimum_compliance = find_skill_compliance(skill_embedding, vector_soft_skills_matrix, SOFT_SKILLS_SIMILARITY_THRESHOLD)
    admissible_jobs_ids = find_job_matches(minimum_compliance[0], market_soft_skills_df)
    admissible_matches_count = minimum_compliance[2]
    
    ideal_compliance = find_skill_compliance(skill_embedding, vector_soft_skills_matrix, SOFT_SKILLS_SIMILARITY_THRESHOLD, weight_soft_skills_matrix, weight)
    ideal_jobs_ids = find_job_matches(ideal_compliance[0], market_soft_skills_df)
    weights = ideal_compliance[1]
    ideal_matches_count = ideal_compliance[2]

    print(f"Admissible Jobs IDs ({admissible_matches_count}): {admissible_jobs_ids}")
    print(f"Ideal Job IDs ({ideal_matches_count}): {ideal_jobs_ids}\nWeights: {weights}\n")

Admissible Jobs IDs (9): ['2039476314', '2043631830', '2073491174', '2112042115', '2115963207', '2121287479', '2122582129', '2122980329']
Ideal Job IDs (4): ['2122980329', '2043631830', '2073491174', '2115963207']
Weights: [np.float32(0.7), np.float32(0.7), np.float32(0.7), np.float32(0.7)]

Admissible Jobs IDs (11): ['2039476314', '2043631830', '2073491174', '2112042115', '2115963207', '2121016617', '2122582129', '2122980329', '2123414317', '2124334518']
Ideal Job IDs (2): ['2123414317', '2115963207']
Weights: [np.float32(0.6), np.float32(0.7)]

Admissible Jobs IDs (14): ['2039476314', '2043631830', '2073491174', '2112042115', '2115963207', '2121016617', '2121287479', '2122980329', '2123414317']
Ideal Job IDs (2): ['2043631830', '2115963207']
Weights: [np.float32(0.5), np.float32(0.5)]

Admissible Jobs IDs (1): ['2121016617']
Ideal Job IDs (0): []
Weights: []

Admissible Jobs IDs (0): []
Ideal Job IDs (0): []
Weights: []

Admissible Jobs IDs (3): ['2112042115', '2121016617', '21212874

For each hard skill

In [120]:
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.72
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_hard_skills_df.shape[0] - 1

for i in range(0, skills_count):
    weight = candidate_hard_skills_df.iloc[(i, HARD_SKILLS_WEIGHT_COLUMN_INDEX)] 
    skill_embedding = candidate_hard_skills_df.iloc[(i, HARD_SKILLS_STRING_COLUMN_INDEX) ] 

    minimum_compliance = find_skill_compliance(skill_embedding, vector_hard_skills_matrix, HARD_SKILLS_SIMILARITY_THRESHOLD)
    admissible_jobs_ids = find_job_matches(minimum_compliance[0], market_hard_skills_df)
    admissible_matches_count = minimum_compliance[2]
    
    ideal_compliance = find_skill_compliance(skill_embedding, vector_hard_skills_matrix, HARD_SKILLS_SIMILARITY_THRESHOLD, weight_hard_skills_matrix, weight)
    ideal_jobs_ids = find_job_matches(ideal_compliance[0], market_hard_skills_df)
    weights = ideal_compliance[1]
    ideal_matches_count = ideal_compliance[2]

    print(f"Admissible Jobs IDs ({admissible_matches_count}): {admissible_jobs_ids}")
    print(f"Ideal Job IDs ({ideal_matches_count}): {ideal_jobs_ids}\nWeights: {weights}\n")

Admissible Jobs IDs (6): ['2039476314', '2123414317', '2073491174', '2112042115']
Ideal Job IDs (6): ['2039476314', '2123414317', '2073491174', '2112042115']
Weights: [np.float32(0.9), np.float32(0.8), np.float32(0.7), np.float32(0.9), np.float32(0.9), np.float32(0.8)]



Admissible Jobs IDs (4): ['2122980329', '2043631830', '2112042115', '2121016617']
Ideal Job IDs (1): ['2121016617']
Weights: [np.float32(0.6)]

Admissible Jobs IDs (9): ['2073491174', '2112042115', '2121016617', '2121287479', '2122980329', '2123414317']
Ideal Job IDs (5): ['2122980329', '2121016617', '2121287479']
Weights: [np.float32(0.6), np.float32(0.6), np.float32(0.6), np.float32(0.6), np.float32(0.5)]

Admissible Jobs IDs (1): ['2039476314']
Ideal Job IDs (0): []
Weights: []

Admissible Jobs IDs (1): ['2112042115']
Ideal Job IDs (0): []
Weights: []

Admissible Jobs IDs (8): ['2039476314', '2043631830', '2073491174', '2115963207', '2122980329', '2123414317']
Ideal Job IDs (0): []
Weights: []

Admissible Jobs IDs (3): ['2112042115']
Ideal Job IDs (1): ['2112042115']
Weights: [np.float32(0.7)]

Admissible Jobs IDs (1): ['2112042115']
Ideal Job IDs (0): []
Weights: []

Admissible Jobs IDs (6): ['2039476314', '2043631830', '2073491174', '2112042115', '2122980329', '2123414317']
Ideal 

### Find candidate's missing skills for market 